# 5. Input files

A function sent to a cluster takes its code and its arguments with it. The
files it opens are another matter: `pd.read_csv("data/subjects.csv")` works on
your computer and raises `FileNotFoundError` on the cluster, because the file
is not there.

For the `ssh` and `slurm` backends clustrix handles the common cases
automatically. Before a job is submitted, it looks for the local files the
call names and makes each one readable by the job. This tutorial shows

- the three ways a file can be named and what happens for each;
- what "in place" means on a shared filesystem;
- where uploaded files are kept;
- `data=[...]` and `no_data=[...]`, and the size limit;
- what automatic staging cannot do, including **output files, which are not
  brought back**;
- declared data packages, for when you want to be explicit.

**About the outputs.** Produced on a single-node test cluster on the same
computer as the notebook (`clx@127.0.0.1:2222`), using the `ssh` backend
because it is quicker; `slurm` behaves the same. The notebook's home directory
was made unreadable to the cluster user, as your laptop's disk is, and
`/tmp/lab-share`, which both users can read, plays the part of a filesystem
that both sides mount.

In [1]:
# cluster-required: a host you can reach with ssh
import json
import logging
import os
from pathlib import Path

import numpy as np

import clustrix
from clustrix import cluster, cluster_du, cluster_ls, configure, get_config

logging.basicConfig(level=logging.WARNING, format="%(message)s")
logging.getLogger("clustrix.auto_stage").setLevel(logging.INFO)   # report each input

configure(
    cluster_type="ssh",
    cluster_host="127.0.0.1",
    cluster_port=2222,
    username="clx",
    key_file="~/.ssh/id_ed25519",
    remote_work_dir="~/clustrix-tutorial",
    replicate_local_environment=False,
    max_parallel_jobs=3,
)
print("working directory:", os.getcwd())

working directory: /home/researcher/analysis


Some data to work with: six small tables, one per subject, and a settings
file.

In [2]:
rng = np.random.default_rng(7)
Path("data/subjects").mkdir(parents=True, exist_ok=True)
for subject in range(1, 7):
    np.savetxt(f"data/subjects/sub-{subject:02d}.csv", rng.normal(size=(400, 6)), delimiter=",")
Path("data/thresholds.json").write_text(json.dumps({"low": -1.0, "high": 1.0}))

for item in sorted(Path("data").rglob("*")):
    if item.is_file():
        print(f"{item.stat().st_size:>7} bytes  {item}")

  61277 bytes  data/subjects/sub-01.csv
  61191 bytes  data/subjects/sub-02.csv
  61203 bytes  data/subjects/sub-03.csv
  61183 bytes  data/subjects/sub-04.csv
  61212 bytes  data/subjects/sub-05.csv
  61178 bytes  data/subjects/sub-06.csv
     26 bytes  data/thresholds.json


## A file named in the code

The simplest case: the path is a string in the function.

In [3]:
@cluster
def read_thresholds():
    import json
    import os

    with open("data/thresholds.json") as handle:
        thresholds = json.load(handle)
    return thresholds, os.path.realpath("data/thresholds.json")


thresholds, seen_at = read_thresholds()
print(thresholds)
print("the job read:", seen_at)

Input /home/researcher/analysis/data/thresholds.json (26 B): uploaded as /home/clx/clustrix-tutorial/data/f83b28395cbcfee34cc2/thresholds.json


{'low': -1.0, 'high': 1.0}
the job read: /home/clx/clustrix-tutorial/data/f83b28395cbcfee34cc2/thresholds.json


The log line says what was done: the file was **uploaded**, into a directory
called `data` under `remote_work_dir`. In the job's working directory a link
named `data/thresholds.json` pointed at that copy, so the code ran unchanged
with its relative path.

Clustrix finds such paths among the string constants of the decorated
function, of the functions from your project that it calls, and of the global
variables they read. A string counts only if a file or directory of that name
exists on this machine.

A second call uploads nothing:

In [4]:
thresholds, _ = read_thresholds()

Input /home/researcher/analysis/data/thresholds.json (26 B): already uploaded as /home/clx/clustrix-tutorial/data/f83b28395cbcfee34cc2/thresholds.json


## A directory passed as an argument

Paths among the arguments are found too, at any depth inside lists, tuples,
sets and dicts. A directory is taken with what is in it, except for
files that look like credentials (each is named in a warning), symbolic
links, anything listed in `no_data`, and `.git`, `__pycache__` and
`.ipynb_checkpoints` directories. This function
also has a loop that can be divided, so each chunk becomes a job and each job
gets the link.

In [5]:
@cluster
def subject_statistics(folder):
    import os

    import numpy as np

    statistics = {}
    for name in sorted(os.listdir(folder)):
        table = np.loadtxt(os.path.join(folder, name), delimiter=",")
        statistics[name] = (round(float(table.mean()), 4), table.shape)
    return statistics


for name, (mean, shape) in subject_statistics("data/subjects").items():
    print(name, mean, shape)

Input /home/researcher/analysis/data/subjects (358.6 KB): uploaded as /home/clx/clustrix-tutorial/data/df927dedeada4885a99d/subjects


sub-01.csv -0.0354 (400, 6)
sub-02.csv -0.0085 (400, 6)
sub-03.csv -0.015 (400, 6)
sub-04.csv 0.0033 (400, 6)
sub-05.csv -0.002 (400, 6)
sub-06.csv 0.017 (400, 6)


A relative path stays as it is and is linked into the job directory. A bare
word such as `"train"` is treated as a path only when the parameter it is
passed to has a name like `path`, `file`, `folder`, `dir`, `data`, `input` or
`root`, so `mode="train"` does not upload a directory that happens to be
called `train`.

## An absolute path passed as an argument

An absolute local path means nothing on the cluster, and a link inside the job
directory cannot fix that. So the file is uploaded and **the argument is
rewritten** to the cluster's copy.

In [6]:
@cluster
def count_rows(table_path):
    with open(table_path) as handle:
        return table_path, sum(1 for _ in handle)


local_path = str(Path("data/subjects/sub-01.csv").resolve())
print("passed:  ", local_path)

received, rows = count_rows(local_path)
print("received:", received)
print("rows:    ", rows)

passed:   /home/researcher/analysis/data/subjects/sub-01.csv


Input /home/researcher/analysis/data/subjects/sub-01.csv (59.8 KB): uploaded as /home/clx/clustrix-tutorial/data/6463afcbef766e241cdf/sub-01.csv


received: /home/clx/clustrix-tutorial/data/6463afcbef766e241cdf/sub-01.csv
rows:     400


An absolute path written *in the code* cannot be rewritten that way, since
clustrix does not edit your function. The file is still uploaded, and a
warning tells you to pass the path as an argument instead.

## "In place": files the cluster already has

If the same absolute path exists on the cluster with the same size, nothing
is copied and nothing is rewritten. That is the situation on a filesystem
mounted on both sides (a lab share, or a notebook that itself runs on the
cluster's login node), and for data you have already put on the cluster at
the same path.

The test cluster is on the same computer as this notebook, so a directory
both users can read stands in for such a filesystem here. On a real cluster
that does not mount your disk, this same call would upload the file.

In [7]:
shared = Path("/tmp/lab-share/reference")        # stands in for a mounted share
shared.mkdir(parents=True, exist_ok=True)
np.savetxt(shared / "atlas.csv", np.arange(600).reshape(200, 3), fmt="%d", delimiter=",", header="x,y,z")

received, rows = count_rows(str(shared / "atlas.csv"))
print("received:", received, "| rows:", rows)

Input /tmp/lab-share/reference/atlas.csv (2.2 KB): in place


received: /tmp/lab-share/reference/atlas.csv | rows: 201


If the path exists on both sides but the sizes differ, clustrix stops with an
error instead of guessing which file you meant.

## The store

Uploaded inputs live under `<remote_work_dir>/data/`, in one directory per
distinct content.

In [8]:
store = os.path.dirname(os.path.dirname(seen_at))
print(store)
for entry in sorted(cluster_ls(store, get_config())):
    print("   ", entry, "->", cluster_ls(f"{store}/{entry}", get_config()))

/home/clx/clustrix-tutorial/data


    6463afcbef766e241cdf -> ['.complete', 'sub-01.csv']
    df927dedeada4885a99d -> ['.complete', 'subjects']


    f83b28395cbcfee34cc2 -> ['.complete', 'thresholds.json']


(`.complete` is clustrix's mark that an upload finished; a copy is used only
once it is there.) The directory names are derived from the content, so:

- the same file is uploaded once, however many calls and sessions use it;
- if you edit a file, the next call uploads the new version next to the old
  one, and jobs already running keep reading what they started with;
- the copies are read-only, so a job cannot change what another job reads.

In [9]:
Path("data/thresholds.json").write_text(json.dumps({"low": -2.0, "high": 2.0}))
thresholds, seen_at_now = read_thresholds()
print(thresholds, "| a different copy:", seen_at_now != seen_at)

Input /home/researcher/analysis/data/thresholds.json (26 B): uploaded as /home/clx/clustrix-tutorial/data/d0d217a36bf1bec48b0e/thresholds.json


{'low': -2.0, 'high': 2.0} | a different copy: True


A job that tries to write into a staged input gets `PermissionError`. Your
file here is not touched either way; the job only ever sees the copy.

In [10]:
@cluster
def overwrite(path):
    with open(path, "w") as handle:
        handle.write("{}")


try:
    overwrite("data/thresholds.json")
except PermissionError as error:
    print("PermissionError:", error)

print("here, unchanged:", Path("data/thresholds.json").read_text())

Input /home/researcher/analysis/data/thresholds.json (26 B): already uploaded as /home/clx/clustrix-tutorial/data/d0d217a36bf1bec48b0e/thresholds.json


PermissionError: [Errno 13] Permission denied: 'data/thresholds.json'
here, unchanged: {"low": -2.0, "high": 2.0}


**Nothing in the store is ever deleted automatically.** `cleanup_on_success`
removes job directories, not data. Check what it holds, and remove it over SSH
when a project is finished. Because the copies are read-only, make them
writable first:

```bash
ssh -p 2222 clx@127.0.0.1 'chmod -R u+w ~/clustrix-tutorial/data && rm -rf ~/clustrix-tutorial/data'
```

In [11]:
usage = cluster_du(store, get_config())
print(f"{usage.file_count} files, {usage.total_bytes / 1e6:.2f} MB in {store}")

13 files, 0.43 MB in /home/clx/clustrix-tutorial/data


## Paths clustrix cannot see: `data=[...]`

A path that is computed while the function runs names nothing beforehand.
This function builds its file name from an argument:

In [12]:
@cluster
def subject_mean(subject):
    import numpy as np

    table = np.loadtxt(f"data/subjects/sub-{subject:02d}.csv", delimiter=",")
    return round(float(table.mean()), 4)


try:
    subject_mean(3)
except FileNotFoundError as error:
    print("FileNotFoundError:", error)

FileNotFoundError: data/subjects/sub-03.csv not found.


The string in the code is a template, not a path, so nothing was staged and
the job failed as it would have without clustrix. The same goes for `glob`
patterns, `os.path.join` of variables, and paths read from a configuration
file.

Name such inputs yourself with `data=[...]` on the decorator. Each entry is
staged as if it had been passed as an argument.

In [13]:
@cluster(data=["data/subjects"])
def subject_mean(subject):
    import numpy as np

    table = np.loadtxt(f"data/subjects/sub-{subject:02d}.csv", delimiter=",")
    return round(float(table.mean()), 4)


print(subject_mean(3))

Input /home/researcher/analysis/data/subjects (358.6 KB): already uploaded as /home/clx/clustrix-tutorial/data/df927dedeada4885a99d/subjects


-0.015


The other fix is to pass the directory as an argument, as
`subject_statistics` does. That is usually the better design anyway: the
function no longer depends on where it is called from.

## Keeping a path out: `no_data=[...]`

The opposite case: the code names a local path that the job should *not* be
given, for example a local cache that the cluster job will rebuild.

In [14]:
Path("cache").mkdir(exist_ok=True)
Path("cache/features.npy").write_bytes(b"\0" * 200_000)


@cluster(no_data=["cache/features.npy"])
def uses_cache():
    import os

    return os.path.exists("cache/features.npy")


print("cache present in the job:", uses_cache())

cache present in the job: False


No input was reported, and the job did not have the file.

## The size limit

Automatic staging is meant for the files a script casually opens, not for a
dataset. If one call names more than `auto_stage_max_bytes` of local input
that would have to be uploaded (default: 1 GB), the call stops before copying
anything and lists what it found. The limit is lowered here to show the
message.

In [15]:
from clustrix import StagingError

configure(auto_stage_max_bytes=100_000)


@cluster
def needs_cache():
    return os.path.getsize("cache/features.npy")


try:
    needs_cache()
except StagingError as error:
    print(error)

configure(auto_stage_max_bytes=1_000_000_000)

This call names 195.3 KB of local input, more than clustrix copies to a cluster automatically (auto_stage_max_bytes = 97.7 KB):
    195.3 KB  /home/researcher/analysis/cache/features.npy  (the code of needs_cache)
Put large data on the cluster yourself and pass its path there, raise auto_stage_max_bytes, or exclude a path with no_data=[...].


For data of real size, the right place is the cluster's own storage: copy it
there once with `rsync` or your site's transfer tool, and pass the cluster
path as an argument. A path that does not exist on your computer is left
alone and simply used by the job.

`configure(auto_stage=False)` turns automatic staging off entirely.

## What automatic staging will not do

### It will not take files from outside your project because code names them

A path found *in code* is staged only if it lies inside the project: the git
repository the working directory belongs to, or the working directory itself
when there is no repository. Symbolic links are resolved first. Code you
imported from elsewhere, or a stray string, cannot cause an arbitrary file on
your computer to be uploaded.

A path you pass as an **argument** or list in `data=[...]` is your explicit
choice and may be anywhere. (The constant in the second cell below is this
notebook's home directory written out; yours differs.)

In [16]:
notes = Path.home() / "private" / "notes.txt"
notes.parent.mkdir(exist_ok=True)
notes.write_text("not part of the project\n")
print(notes, "| project:", os.getcwd())

/home/researcher/private/notes.txt | project: /home/researcher/analysis


In [17]:
@cluster
def read_by_literal():
    return open("/home/researcher/private/notes.txt").read()


try:
    read_by_literal()
except OSError as error:
    print("named in code:        ", type(error).__name__ + ":", error)


@cluster
def read_by_argument(path):
    return open(path).read()


print("passed as an argument:", repr(read_by_argument(str(notes))))

named in code:         PermissionError: [Errno 13] Permission denied: '/home/researcher/private/notes.txt'


Input /home/researcher/private/notes.txt (24 B): uploaded as /home/clx/clustrix-tutorial/data/fcfc52311fa47f1dade3/notes.txt


passed as an argument: 'not part of the project\n'


The file named only in code was not sent. (On a real cluster the error is
`FileNotFoundError`. The test cluster shares a disk with this notebook, so the
path exists there but belongs to another user.)



### It will not copy a whole project or home directory

The project directory, the working directory, your home directory and
anything above them are refused, as a constant in the code and as an
argument alike. A slip such as passing `"."` would otherwise upload
everything you have.

In [18]:
@cluster
def count_entries(folder):
    import os

    return len(os.listdir(folder))


try:
    count_entries(os.getcwd())
except OSError as error:
    print(type(error).__name__ + ":", error)

Not staging: /home/researcher/analysis (argument 'folder') is the project directory, and clustrix does not copy one of those to a cluster; name the files or directories the job needs (as arguments or in data=[...])


PermissionError: [Errno 13] Permission denied: '/home/researcher/analysis'


The warning says what to do instead: name the files or directories the job
needs. (As above, the error on a real cluster is `FileNotFoundError`.)

### It never uploads credentials

Private keys, token files and similar are refused even when passed
explicitly, with a warning. Inside a directory that is staged, they are left
out and named.

In [19]:
@cluster
def can_read(path):
    import os

    return os.path.exists(path)


print("key readable by the job:", can_read(os.path.expanduser("~/.ssh/id_ed25519")))

Not staging: /home/researcher/.ssh/id_ed25519 looks like a credential and is never uploaded


key readable by the job: False


### Nothing the cluster says can widen this

What is staged is decided on your computer, from your call, before the job
exists. There is no mechanism by which a job can ask for a file.

### Output files are not brought back

Only the function's **return value** returns. A file the job writes stays on
the cluster, and if it was written into the job's working directory it is
deleted with that directory when the job succeeds.

In [20]:
@cluster
def writes_a_file():
    import os

    with open("summary.txt", "w") as handle:
        handle.write("mean = 0.42\n")
    return os.path.abspath("summary.txt")


written = writes_a_file()
print("written on the cluster as:", written)
print("exists here:              ", os.path.exists("summary.txt"))
print("still on the cluster:     ", clustrix.cluster_exists(written, get_config()))

written on the cluster as: /home/clx/clustrix-tutorial/job_1795775211_c9725577/summary.txt
exists here:               False
still on the cluster:      False


It is in neither place: never here, and removed there with the job directory.
There are two patterns for results.

**Return the data.** Arrays, data frames, dictionaries, fitted models, and
the bytes of a figure all travel as return values. Write the file here.

In [21]:
@cluster
def summarise(folder):
    import os

    import numpy as np

    rows = []
    for name in sorted(os.listdir(folder)):
        table = np.loadtxt(os.path.join(folder, name), delimiter=",")
        rows.append((name, float(table.mean()), float(table.std())))
    return rows


Path("results").mkdir(exist_ok=True)
with open("results/summary.csv", "w") as handle:
    handle.write("subject,mean,std\n")
    for name, mean, std in summarise("data/subjects"):
        handle.write(f"{name},{mean:.4f},{std:.4f}\n")

print(open("results/summary.csv").read())

Input /home/researcher/analysis/data/subjects (358.6 KB): already uploaded as /home/clx/clustrix-tutorial/data/df927dedeada4885a99d/subjects


subject,mean,std
sub-01.csv,-0.0354,0.9887
sub-02.csv,-0.0085,0.9983
sub-03.csv,-0.0150,0.9887
sub-04.csv,0.0033,0.9847
sub-05.csv,-0.0020,1.0273
sub-06.csv,0.0170,0.9763



**For large outputs, write to a directory on the cluster that you choose, and
return the path.** Use a location outside the job directory (project or
scratch storage). Fetch the files yourself when you need them.

In [22]:
@cluster
def simulate(output_dir, steps):
    import os

    import numpy as np

    output_dir = os.path.expanduser(output_dir)
    os.makedirs(output_dir, exist_ok=True)
    target = os.path.join(output_dir, f"trajectory-{steps}.npy")
    np.save(target, np.random.default_rng(0).normal(size=(steps, 3)).cumsum(axis=0))
    return target


remote_file = simulate("~/study-results", 50_000)
info = clustrix.cluster_stat(remote_file, get_config())
print(remote_file, "-", info.size, "bytes on the cluster")

/home/clx/study-results/trajectory-50000.npy - 1200128 bytes on the cluster


The `cluster_*` helpers (`cluster_ls`, `cluster_stat`, `cluster_exists`,
`cluster_glob`, `cluster_du`, ...) inspect remote files; they do not transfer
them. To copy the file here, use `scp` or `rsync` from a terminal:

```bash
scp -P 2222 clx@127.0.0.1:study-results/trajectory-50000.npy results/
rsync -av -e "ssh -p 2222" clx@127.0.0.1:study-results/ results/
```

`"~/study-results"` was not staged or rewritten because no such directory
exists on this computer; a path that exists only on the cluster is passed
through untouched.

### Where it does not apply

Automatic staging is done for `ssh` and `slurm`. It is not done for calls
with `async_submit=True`, nor for Hugging Face Jobs. Use a data package there.

## Declared data: `data_package`

When you would rather state exactly which files travel, build a package and
pass it as an argument. On the worker, ask the package for the path of a
file.

In [23]:
package = clustrix.data_package(["data/thresholds.json", "data/subjects/sub-01.csv"], name="pilot")
print(package)


@cluster
def read_package(pkg):
    import json

    with open(pkg.path("thresholds.json")) as handle:
        thresholds = json.load(handle)
    with open(pkg.path("subjects/sub-01.csv")) as handle:
        rows = sum(1 for _ in handle)
    return thresholds, rows


print(read_package(package))

DataPackage(name='pilot', files=2, bytes=61303, at=inline)


({'low': -2.0, 'high': 2.0}, 400)


The paths inside a package are relative to the common parent of the files
named (here `data/`). A package this small travels inside the function's
arguments ("inline"); nothing is stored anywhere else.

A package larger than `stage_inline_max_bytes` (1 MB) is uploaded to a
**private dataset repository in your Hugging Face account**, which clustrix
creates. That suits Hugging Face Jobs. An `ssh` or `slurm` worker cannot read
it, and clustrix raises a clear error in that case; for those backends use
automatic staging or the cluster's own storage. Like the store above, a
package's remote copy is never deleted automatically: `package.delete()`,
`clustrix.list_data_packages()` and `clustrix.delete_data_package(id)` manage
it. The reference page [Data packages](../data_packages.rst) has the details.

## Tidying up

This notebook created two things outside its working directory on this
computer. On the cluster it left the store, the environments and
`~/study-results`, which are yours to remove as described above.

In [24]:
import shutil

shutil.rmtree("/tmp/lab-share")
notes.unlink()
if not any(notes.parent.iterdir()):      # leave a directory that holds anything else
    notes.parent.rmdir()
print(os.path.exists("/tmp/lab-share"), notes.exists())

False False


## If something looks wrong

| You see | It means |
|---|---|
| `FileNotFoundError` in the remote traceback | The path was computed at run time, or lies outside the project and was named only in code. Pass it as an argument or add `data=[...]`. |
| `This call names ... of local input, more than clustrix copies to a cluster automatically` | The inputs exceed `auto_stage_max_bytes`. Put the data on the cluster and pass that path, or exclude a path with `no_data=[...]`. |
| `... exists on the cluster too, but as a different file` | Same absolute path on both sides, different sizes. Remove or rename one, or use `no_data=[...]` to use the cluster's file. |
| `Not staging: ... looks like a credential and is never uploaded` | A key or token file was passed. It will not be sent. |
| `... was copied to the cluster as ..., but the code names it by an absolute path` | An absolute path is written in the code. Pass it as an argument so that it can be pointed at the copy. |
| An input you did not expect is reported | The log names where each path was found. A global variable is considered when the function's code mentions its name, and that includes attribute names: a notebook variable called `path` matches `os.path`. Rename the variable, or exclude the file with `no_data=[...]`. |
| `Not staging: ... is the project directory` (or `the home directory`) | A whole project or home directory was named. Name the files or directories the job needs. |
| `PermissionError` when a job writes into an input | Staged copies are read-only. Write to a directory of your own on the cluster. |
| `rm -rf` of the store fails with `Permission denied` | Run `chmod -R u+w <remote_work_dir>/data` first. |
| An output file is missing | Outputs are not brought back. Return the data, or write to a cluster directory you fetch from. |
| The remote directory keeps growing | `<remote_work_dir>/data` and `envs` are never cleaned automatically. |

## Next

[Tutorial 6](06_notebook_widget.ipynb): configuring all of this from a
notebook widget.